# Crude Supply-Demand Balance & Inventory Shock Study

Objective: evaluate whether EIA inventory and Cushing shocks transmit into WTI volatility and the Brent-WTI spread. The analysis combines weekly physical petroleum balances with daily spot-market outcomes. This is **not** price prediction.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()

from build_balance import build_crude_balance
from build_features import FUNDAMENTAL_FEATURES, PRICE_FEATURES, build_features, compute_shock_zscore
from event_study import run_all_event_studies, run_event_study
from fetch_eia import fetch_all_eia
from fetch_prices import fetch_daily_prices
from figures import (
    plot_extreme_bar,
    plot_event_study_comparison,
    plot_inventory_seasonality,
    plot_price_history,
    plot_spread,
)
from generate_memo import generate_all_historical_memos, generate_memo
from risk_overlay import run_risk_overlay

In [2]:
prices = fetch_daily_prices(start="2010-01-01")
print(prices.shape)
print(prices.index.min(), prices.index.max())
prices.tail(5)

Daily prices: 4,130 rows, 2010-01-04 to 2026-06-22
(4130, 2)
2010-01-04 00:00:00 2026-06-22 00:00:00


,wti_price,brent_price
date,,
2026-06-15,84.65,84.36
2026-06-16,79.80,80.50
2026-06-17,80.65,80.33
2026-06-18,80.35,79.35
2026-06-22,78.94,76.49


In [3]:
eia = fetch_all_eia(start="2010-01-01")
print(eia.shape)
print(eia.index.min(), eia.index.max())
eia.tail(5)

WCESTUS1 -> us_crude_stocks: 860 rows, 2010-01-01 to 2026-06-19


W_EPC0_SAX_YCUOK_MBBL -> cushing_stocks: 860 rows, 2010-01-01 to 2026-06-19


WCRFPUS2 -> us_crude_production: 860 rows, 2010-01-01 to 2026-06-19


WCRIMUS2 -> crude_imports: 860 rows, 2010-01-01 to 2026-06-19


WCREXUS2 -> crude_exports: 860 rows, 2010-01-01 to 2026-06-19


WGIRIUS2 -> refinery_inputs: 860 rows, 2010-01-01 to 2026-06-19


WPULEUS3 -> refinery_utilization: 860 rows, 2010-01-01 to 2026-06-19
Merged EIA weekly: 860 rows, 2010-01-01 to 2026-06-19
(860, 7)
2010-01-01 00:00:00 2026-06-19 00:00:00


,us_crude_stocks,cushing_stocks,us_crude_production,crude_imports,crude_exports,refinery_inputs,refinery_utilization
week_end,,,,,,,
2026-05-22,441686,23024,13715,5212,4440,17171,94.5
2026-05-29,433712,22441,13707,6397,5874,17199,94.7
2026-06-05,426485,21640,13799,5888,4840,17315,95.3
2026-06-12,418222,20034,13806,5134,4327,17415,96.7
2026-06-19,412134,18957,13819,5570,4669,17317,96.1


In [4]:
balance = build_crude_balance(eia)
balance[[
    "crude_supply_kbd",
    "crude_demand_kbd",
    "crude_supply_kbbl_week",
    "crude_demand_kbbl_week",
    "implied_stock_change_kbbl",
    "crude_balance",
    "inventory_change",
    "balance_residual",
]].describe()

Crude balance: 859 rows, 2010-01-08 to 2026-06-19


,crude_supply_kbd,crude_demand_kbd,crude_supply_kbbl_week,crude_demand_kbbl_week,implied_stock_change_kbbl,crude_balance,inventory_change,balance_residual
count,859.000000,859.000000,859.000000,859.000000,859.000000,859.000000,859.000000,859.000000
mean,17263.165308,17990.409779,120842.157159,125932.868452,-5090.711292,-5090.711292,122.826542,-5213.537835
std,1882.100593,2205.816567,13174.704154,15440.715972,6315.713256,6315.713256,4989.554320,4482.045573
min,13288.000000,12651.000000,93016.000000,88557.000000,-27419.000000,-27419.000000,-17049.000000,-23972.000000
25%,15708.500000,16052.500000,109959.500000,112367.500000,-8326.500000,-8326.500000,-3265.000000,-7156.500000
50%,17238.000000,17832.000000,120666.000000,124824.000000,-4557.000000,-4557.000000,313.000000,-4535.000000
75%,18975.000000,20033.000000,132825.000000,140231.000000,-1057.000000,-1057.000000,3046.500000,-2617.000000
max,21504.000000,23073.000000,150528.000000,161511.000000,25487.000000,25487.000000,21563.000000,8860.000000


The weekly physical balance is approximated as crude supply minus crude demand, where supply is domestic production plus imports and demand is exports plus refinery inputs. EIA flow series are reported in thousand barrels per day, so daily rates are converted to weekly volumes before comparing implied stock change with reported inventory change. The residual checks whether the market story is grounded in physical barrels rather than only prices.

In [5]:
merged = build_features(balance, prices)
feature_names = [c for c in FUNDAMENTAL_FEATURES + PRICE_FEATURES if c in merged.columns]
print(len(feature_names), feature_names)
print(merged.shape)
merged.tail(5)

Merged event table: 859 rows, 2010-01-08 to 2026-06-19
Feature count: 20
Features: inventory_change, cushing_change, production_change, net_import_change, refinery_util_change, crude_balance, balance_residual, inventory_shock_z, cushing_shock_z, production_shock_z, days_of_supply_z, top_decile_inventory_draw, top_decile_inventory_build, top_decile_cushing_draw, top_decile_cushing_build, brent_wti_spread, spread_zscore, trailing_20d_vol, wti_return, fwd_5d_realized_vol
20 ['inventory_change', 'cushing_change', 'production_change', 'net_import_change', 'refinery_util_change', 'crude_balance', 'balance_residual', 'inventory_shock_z', 'cushing_shock_z', 'production_shock_z', 'days_of_supply_z', 'top_decile_inventory_draw', 'top_decile_inventory_build', 'top_decile_cushing_draw', 'top_decile_cushing_build', 'brent_wti_spread', 'spread_zscore', 'trailing_20d_vol', 'wti_return', 'fwd_5d_realized_vol']
(859, 45)


,us_crude_stocks,cushing_stocks,us_crude_production,crude_imports,crude_exports,refinery_inputs,refinery_utilization,inventory_change,inventory_change_kbbl,cushing_change,...,wti_price,brent_price,brent_wti_spread,wti_return,trailing_20d_vol,spread_zscore,fwd_5d_realized_vol,fwd_5d_spread_change,fwd_5d_wti_return,date
week_end,,,,,,,,,,,,,,,,,,,,,
2026-05-22,441686,23024,13715,5212,4440,17171,94.5,-3327.0,-3327.0,-2794.0,...,92.35,97.11,4.76,-0.055599,0.643077,NaN,0.396727,-2.83,0.077182,2026-05-27
2026-05-29,433712,22441,13707,6397,5874,17199,94.7,-7974.0,-7974.0,-583.0,...,99.76,101.69,1.93,0.023223,0.595346,NaN,0.381307,0.12,-0.062883,2026-06-03
2026-06-05,426485,21640,13799,5888,4840,17315,95.3,-7227.0,-7227.0,-801.0,...,93.68,95.73,2.05,0.019184,0.543334,NaN,0.420119,-2.37,-0.149766,2026-06-10
2026-06-12,418222,20034,13806,5134,4327,17415,96.7,-8263.0,-8263.0,-1606.0,...,80.65,80.33,-0.32,0.010595,0.542697,NaN,NaN,NaN,NaN,2026-06-17
2026-06-19,412134,18957,13819,5570,4669,17317,96.1,-6088.0,-6088.0,-1077.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT


Release-date alignment maps each weekly EIA observation to the first available trading date on or after an approximate release date of three business days after the week end. Daily forward outcomes are computed before the merge. That ordering keeps the event row aligned to information available at or after the estimated release timestamp and avoids using pre-release prices.

In [6]:
plot_price_history(prices)

WindowsPath('outputs/figures/01_price_history.png')

In [7]:
plot_spread(prices)

WindowsPath('outputs/figures/02_brent_wti_spread.png')

In [8]:
plot_inventory_seasonality(balance)

WindowsPath('outputs/figures/03_inventory_seasonality.png')

In [9]:
results = run_all_event_studies(merged)
results

                                           title         shock_col          outcome_col                                            study  n_draws  n_builds  mean_outcome_after_draw  mean_outcome_after_build  difference  t_statistic  p_value  significant_5pct
 Inventory Draw vs Build -> WTI 5-day Volatility inventory_shock_z  fwd_5d_realized_vol  Inventory Draw vs Build -> WTI 5-day Volatility       84        84                 0.310205                  0.389498   -0.079292    -1.606938 0.110769             False
Cushing Draw vs Build -> Brent-WTI Spread Change   cushing_shock_z fwd_5d_spread_change Cushing Draw vs Build -> Brent-WTI Spread Change       83        83                 0.056145                  0.212651   -0.156506    -0.550525 0.582749             False
     Inventory Draw vs Build -> WTI 5-day Return inventory_shock_z    fwd_5d_wti_return      Inventory Draw vs Build -> WTI 5-day Return       84        84                -0.007016                  0.002219   -0.009235    -

,title,shock_col,outcome_col,study,n_draws,n_builds,mean_outcome_after_draw,mean_outcome_after_build,difference,t_statistic,p_value,significant_5pct
0,Inventory Draw vs Build -> WTI 5-day Volatility,inventory_shock_z,fwd_5d_realized_vol,Inventory Draw vs Build -> WTI 5-day Volatility,84,84,0.310205,0.389498,-0.079292,-1.606938,0.110769,False
1,Cushing Draw vs Build -> Brent-WTI Spread Change,cushing_shock_z,fwd_5d_spread_change,Cushing Draw vs Build -> Brent-WTI Spread Change,83,83,0.056145,0.212651,-0.156506,-0.550525,0.582749,False
2,Inventory Draw vs Build -> WTI 5-day Return,inventory_shock_z,fwd_5d_wti_return,Inventory Draw vs Build -> WTI 5-day Return,84,84,-0.007016,0.002219,-0.009235,-0.851415,0.395906,False


In [10]:
plot_extreme_bar(
    merged,
    "inventory_shock_z",
    "fwd_5d_realized_vol",
    "Inventory Shock and Forward WTI Volatility",
    "Annualized realized volatility",
    "04_inventory_shock_volatility.png",
)
fig = plot_event_study_comparison(
    merged,
    "inventory_shock_z",
    "fwd_5d_realized_vol",
    "5-Day WTI Realized Volatility After Extreme Inventory Shocks",
    "Annualized Volatility",
)
plt.close(fig)

In [11]:
plot_extreme_bar(
    merged,
    "cushing_shock_z",
    "fwd_5d_spread_change",
    "Cushing Shock and Forward Brent-WTI Spread",
    "5-day spread change, USD per barrel",
    "05_cushing_shock_spread.png",
)
fig = plot_event_study_comparison(
    merged,
    "cushing_shock_z",
    "fwd_5d_spread_change",
    "5-Day Brent-WTI Spread Change After Extreme Cushing Shocks",
    "Spread Change ($/bbl)",
)
plt.close(fig)

In [12]:
key_features = [
    "inventory_change",
    "inventory_shock_z",
    "cushing_shock_z",
    "implied_stock_change_kbbl",
    "crude_balance",
    "balance_residual",
    "days_of_supply",
    "weeks_of_supply",
    "brent_wti_spread",
    "trailing_20d_vol",
    "fwd_5d_realized_vol",
    "fwd_5d_spread_change",
    "fwd_5d_wti_return",
]
summary_stats = merged[key_features].describe().T
summary_stats.to_csv(ROOT / "outputs" / "tables" / "summary_statistics.csv")
summary_stats.to_csv(ROOT / "outputs" / "tables" / "feature_summary_stats.csv")
summary_stats

,count,mean,std,min,25%,50%,75%,max
inventory_change,859.0,122.826542,4989.554320,-17049.000000,-3265.000000,3.130000e+02,3046.500000,21563.000000
inventory_shock_z,834.0,-0.007145,1.022905,-3.497054,-0.721794,-9.214713e-03,0.657516,3.356252
cushing_shock_z,834.0,-0.014784,1.042016,-3.153428,-0.742721,-4.761534e-02,0.687125,3.686609
implied_stock_change_kbbl,859.0,-5090.711292,6315.713256,-27419.000000,-8326.500000,-4.557000e+03,-1057.000000,25487.000000
crude_balance,859.0,-5090.711292,6315.713256,-27419.000000,-8326.500000,-4.557000e+03,-1057.000000,25487.000000
balance_residual,859.0,-5213.537835,4482.045573,-23972.000000,-7156.500000,-4.535000e+03,-2617.000000,8860.000000
days_of_supply,859.0,25.911446,3.880926,19.594734,22.846989,2.565422e+01,27.822902,47.049029
weeks_of_supply,859.0,3.701635,0.554418,2.799248,3.263856,3.664889e+00,3.974700,6.721290
brent_wti_spread,856.0,6.022874,5.827338,-5.310000,2.380000,4.500000e+00,7.620000,28.490000
trailing_20d_vol,851.0,0.344189,0.220934,0.110623,0.230968,2.898972e-01,0.401666,2.638635


Inventory draws are negative stock changes and generally indicate a tighter physical crude market, while builds suggest looser conditions. Large draws can matter for volatility because they may force traders to reassess near-term balances quickly. Cushing is especially important for WTI because it is the delivery hub embedded in the benchmark's physical settlement logic. When Cushing inventories tighten, the Brent-WTI spread can react differently from broad U.S. stock changes. The event-study comparisons here are best read as diagnostics of market transmission, not a forecasting model. Statistical significance depends on the sample, the release-date approximation, and the fact that no consensus expectations are included. The results therefore describe historical conditional averages, not a trading rule.

Limitations: the release date is approximated as three business days after week-end, not pulled from the actual EIA holiday-adjusted release calendar. The study does not include survey expectations, so it measures raw shocks rather than surprises relative to market consensus. It does not include the futures curve, options-implied volatility, refinery outages, storage constraints, or macro news controls. Forward returns and volatility are short-horizon realized outcomes, not tradable forecasts. This is not a trading strategy.

## Optional Market Risk Overlay

The fundamentals workflow can be extended into market-risk diagnostics by modeling WTI conditional volatility and one-day 99% VaR. This section is intentionally framed as a risk appendix rather than the core commodity-research result.

In [13]:
risk_results = run_risk_overlay(train_end="2019-12-31", test_start="2020-01-01")
risk_results["comparison"]

WTI returns: 4,128 rows, 2010-01-05 to 2026-06-22, mean=-0.000008, std=0.029865
Risk overlay split: train=2010-01-05 to 2019-12-31 (2,512 obs), test=2020-01-02 to 2026-06-22 (1,616 obs)
GARCH_normal: loglik=-5190.88, AIC=10389.75, BIC=10413.06, params=4
GARCH_t: loglik=-5103.25, AIC=10216.51, BIC=10245.65, params=5
GJR_normal: loglik=-5162.09, AIC=10334.17, BIC=10363.31, params=5
GJR_t: loglik=-5088.52, AIC=10189.03, BIC=10224.01, params=6
Best model: GJR_t (AIC=10189.0)


Backtest: 13 breaches in 1616 days (0.804% vs 1.0% target)
Kupiec p-value: 0.4134


,model,log_likelihood,aic,bic,num_params
0,GJR_t,-5088.517354,10189.034709,10224.005327,6
1,GARCH_t,-5103.252564,10216.505129,10245.647311,5
2,GJR_normal,-5162.085244,10334.170488,10363.312670,5
3,GARCH_normal,-5190.875074,10389.750147,10413.063893,4


In [14]:
pd.DataFrame([risk_results["backtest"]])

,sample,model,train_start,train_end,test_start,test_end,total_observations,num_breaches,breach_rate_pct,expected_rate_pct,LR_statistic,p_value,reject_H0_5pct,interpretation
0,out_of_sample_fixed_parameter,GJR_t,2010-01-05,2019-12-31,2020-01-02,2026-06-22,1616,13,0.804,1.0,0.6689,0.4134,False,Breach rate consistent with 1% target


## Weekly Analyst Memo

A commodity research analyst's last mile is turning market data into concise commentary. The memo generator uses deterministic rules, not LLM text generation, to translate inventory shocks, Cushing tightness, refinery utilization, spread regime, and volatility state into a weekly market note.

In [15]:
latest_memo_path = generate_memo()
historical_paths = generate_all_historical_memos(last_n=4)
latest_memo_path, len(historical_paths)

Memo generated: outputs\memos\weekly_crude_monitor_2026-06-19.md
Overall tone: Bullish
Key risk: Cushing stocks at 19.0 Mb, approaching operational minimum (~22 Mb). Further dra...
  Generated: weekly_crude_monitor_2026-05-29.md - Tone: Bullish
  Generated: weekly_crude_monitor_2026-06-05.md - Tone: Bullish
  Generated: weekly_crude_monitor_2026-06-12.md - Tone: Bullish
  Generated: weekly_crude_monitor_2026-06-19.md - Tone: Bullish


(WindowsPath('outputs/memos/weekly_crude_monitor_2026-06-19.md'), 4)

In [16]:
from IPython.display import Markdown
Markdown(latest_memo_path.read_text(encoding="utf-8"))

# Weekly Crude Market Monitor

**Week Ending:** June 19, 2026
**Approximate Release Date:** June 24, 2026
**Overall Tone:** Bullish

---

## EIA Release Summary

| Metric | Level | Weekly Change | Shock Z-Score |
|--------|-------|--------------|---------------|
| U.S. Crude Stocks | 412.1 Mb | -6088.0 kb | -1.12 |
| Cushing Stocks | 19.0 Mb | -1077.0 kb | -0.95 |
| Production | 13819 kb/d | +13 kb/d | - |
| Refinery Utilization | 96.1% | -0.6 pp | - |
| Net Imports | 901 kb/d | - | - |

## Supply-Demand Balance

Implied weekly stock draw of 18179 kbbl - supply < demand

- Supply (production + imports): 19389 kb/d (135723 kbbl/week)
- Demand (exports + refinery inputs): 21986 kb/d (153902 kbbl/week)
- Implied stock change: -18179 kbbl/week
- Residual vs reported inventory change: -12091 kbbl
- Days of supply: 23.8 days (3.4 weeks)

## Market Assessment

**Tone: Bullish**

- Inventory draw (-6088.0 kb), 1.1 std below seasonal norm
- Refinery utilization high at 96.1% - strong crude demand
- Third consecutive weekly draw

## Spread & Volatility

*Market data shown from latest available aligned price date: June 17, 2026.*

- **WTI:** $80.65/bbl
- **Brent:** $80.33/bbl
- Brent-WTI spread at $-0.32/bbl
- 20-day realized vol at 54.3% - elevated risk environment
- **Volatility Regime:** High

## Key Risk

Cushing stocks at 19.0 Mb, approaching operational minimum (~22 Mb). Further draws could spike WTI basis.

---

*This memo is auto-generated from the Crude Supply-Demand Balance & Inventory
Shock Monitor pipeline. Data sources: EIA Weekly Petroleum Status Report,
FRED (WTI/Brent spot prices). Not investment advice.*
